# Venue ML Coverage Walkthrough — SerpApi Candidate Selection

**SOP Date**: 2026-06-23  
**Directory**: `Data+ML/test/6.22-6.27/`  

**Goal**: Identify which venues have Google Popular Times data (via SerpApi) for supervised ML training.

**Key Constraint**:  
> Search query 用于批量发现 candidates；Place query 只用于最终 label 验证。  
> 不要对每个本地 venue 直接消耗一次 SerpApi 调用。

**Workflow**:
1. Load & prepare venue data
2. Coverage audits (category / district / Citi Bike proximity)
3. Priority scoring for candidate selection
4. SerpApi Search batch discovery
5. Place API validation (limited)
6. Label status & ML candidate list
7. Coverage audit report & output artifacts

In [7]:
# Cell 1: Setup & Imports
import sys, os, json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

PROJECT_ROOT = Path('/Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project')
MODULE_DIR  = PROJECT_ROOT / 'Data+ML' / 'test' / '6.22-6.27' / 'src'
OUTPUT_DIR  = PROJECT_ROOT / 'Data+ML' / 'test' / '6.22-6.27' / 'output'
VENUE_FILE  = PROJECT_ROOT / 'Data+ML' / 'test' / '6.8-6.12_DB' / 'tests' / 'output' / 'venues_clean.csv'
COVERAGE_DETAIL = PROJECT_ROOT / 'Data+ML' / 'test' / '6.15-6.20' / 'output' / 'venue_coverage_detail.csv'

sys.path.insert(0, str(MODULE_DIR))
import venue_serpapi as vs

# SerpApi key — set via env var or leave empty for dry-run mode

os.environ['SERPAPI_API_KEY'] = '2cf9f352dcbaf2ae292abee2718b4c1257342ff3929f7c0925b342fdfda72bb8'
SERPAPI_KEY = os.environ.get('SERPAPI_API_KEY', '')
DRY_RUN = not bool(SERPAPI_KEY)

# Ensure output directories exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'serpapi_raw_responses').mkdir(exist_ok=True)

print(f'Project  : {PROJECT_ROOT}')
print(f'Module   : {MODULE_DIR / "venue_serpapi.py"}')
print(f'Output   : {OUTPUT_DIR}')
print(f'Venues   : {VENUE_FILE}')
print(f'Started  : {datetime.now():%Y-%m-%d %H:%M}')
print(f'SerpApi  : {"LIVE" if not DRY_RUN else "DRY-RUN (no API key)"}')
print(f'Sources  : {len(vs.SERPAPI_SEARCH_CATEGORIES)} categories × {len(vs.DISTRICT_CENTERS)} districts = {len(vs.SERPAPI_SEARCH_CATEGORIES) * len(vs.DISTRICT_CENTERS)} queries')

Project  : /Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project
Module   : /Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/Data+ML/test/6.22-6.27/src/venue_serpapi.py
Output   : /Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/Data+ML/test/6.22-6.27/output
Venues   : /Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/Data+ML/test/6.8-6.12_DB/tests/output/venues_clean.csv
Started  : 2026-06-23 17:44
SerpApi  : LIVE
Sources  : 4 categories × 4 districts = 16 queries


### Cell 1 输出解读

**环境初始化完成**，输出显示：
- **Project**: 项目根目录
- **Module**: 引用的 `venue_serpapi` 模块路径
- **Venues**: 输入数据来源（`venues_clean.csv`）
- **SerpApi**: `DRY-RUN (no API key)` — 当前为离线模式，不会调用真实 SerpApi
- **Sources**: 4个类别 × 4个区域 = 16次搜索查询（每次消耗1次 API 配额）

> 💡 **如终端已设置变量但此处仍显示 DRY-RUN**：取消 Cell 1 中 `os.environ['SERPAPI_API_KEY'] = '...'` 的注释，粘贴你的 Key，然后重新执行 Cell 1 即可切换为 `LIVE`。无需重启内核。


In [8]:
# Cell 2: Load venues
venues, dup_count = vs.load_venues(VENUE_FILE)
print(f'Rows loaded  : {len(venues) + dup_count}')
print(f'Duplicates   : {dup_count}')
print(f'Unique venues: {len(venues)}')
print(f'\nVenue type distribution:')
print(venues['venue_type'].value_counts().to_string())
print(f'\nDistrict distribution:')
print(venues['district'].value_counts().to_string())

# Show in-scope vs out-of-scope
in_scope = venues[~venues['venue_type'].isin(vs.OUT_OF_SCOPE_CATEGORIES)]
print(f'\nIn-scope (for ML): {len(in_scope)}')
print(f'Out-of-scope (AED): {len(venues) - len(in_scope)}')

Rows loaded  : 4838
Duplicates   : 0
Unique venues: 4838

Venue type distribution:
venue_type
emergencyasset    3279
healthcare        1086
restroom           473

District distribution:
district
downtown        1467
midtown_west    1428
midtown_east    1182
uptown           703

In-scope (for ML): 1559
Out-of-scope (AED): 3279


In [9]:
# Cell 3: Load Citi Bike proximity data (from previous coverage run)
citibike_detail = None
if COVERAGE_DETAIL.exists():
    citibike_detail = pd.read_csv(COVERAGE_DETAIL)
    print(f'Citi Bike detail loaded: {len(citibike_detail)} rows')
    print(f'Columns: {list(citibike_detail.columns[:8])}...')
    print(f'\nNearest distance stats:')
    dist_col = 'citibike_nearest_distance_m'
    if dist_col in citibike_detail.columns:
        print(f'  Median: {citibike_detail[dist_col].median():.0f}m')
        print(f'  P90:    {citibike_detail[dist_col].quantile(0.9):.0f}m')
        print(f'  Max:    {citibike_detail[dist_col].max():.0f}m')
        print(f'  ≤200m:  {(citibike_detail[dist_col] <= 200).sum()} ({(citibike_detail[dist_col] <= 200).mean()*100:.1f}%)')
else:
    print('Citi Bike detail file not found — proximity audit will be skipped')

Citi Bike detail loaded: 4838 rows
Columns: ['venue_id', 'venue_type', 'district', 'latitude', 'longitude', 'citibike_nearest_source_id', 'citibike_nearest_distance_m', 'citibike_covered_100m']...

Nearest distance stats:
  Median: 107m
  P90:    192m
  Max:    8654169m
  ≤200m:  4438 (91.7%)


---
## Coverage Audit #1: By Category

In [10]:
# Cell 4: Category coverage audit
cat_audit = vs.audit_coverage_by_category(venues)
print('Category Coverage Audit:')
print(cat_audit.to_string(index=False))
print(f'\nNote: emergencyasset ({cat_audit[cat_audit.category=="emergencyasset"].total_venues.values[0]}) = AEDs, out of scope for ML')

Category Coverage Audit:
      category  total_venues  out_of_scope  ml_eligible  no_data  has_popular_times  checked_count  ml_coverage_pct validation_success_pct
emergencyasset          3279          3279            0        0                  0              0              0.0                   None
    healthcare          1086             0            0     1086                  0              0              0.0                   None
      restroom           473             0            0      473                  0              0              0.0                   None

Note: emergencyasset (3279) = AEDs, out of scope for ML


---
## Coverage Audit #2: By District

In [11]:
# Cell 5: District coverage audit
dist_audit = vs.audit_coverage_by_district(venues)
print('District Coverage Audit:')
print(dist_audit.to_string(index=False))

District Coverage Audit:
    district  total_venues  out_of_scope  ml_eligible  no_data  has_popular_times  checked_count  ml_coverage_pct validation_success_pct
    downtown          1467           901            0      566                  0              0              0.0                   None
midtown_east          1182           786            0      396                  0              0              0.0                   None
midtown_west          1428          1211            0      217                  0              0              0.0                   None
      uptown           703           381            0      322                  0              0              0.0                   None


---
## Coverage Audit #3: Citi Bike Proximity

In [12]:
# Cell 6: Citi Bike proximity audit
cb_audit = vs.audit_citi_bike_proximity(venues, citibike_detail)
if len(cb_audit) > 0:
    print('Citi Bike Proximity Distribution:')
    print(cb_audit.to_string(index=False))
else:
    print('Citi Bike proximity audit skipped (no detail data)')

Citi Bike Proximity Distribution:
   proximity_bucket  total_venues  in_scope_venues  pct_of_total
             0-100m          2199              645          45.5
           100-200m          2239              702          46.3
           200-300m           303              127           6.3
           300-500m            26               25           0.5
              500m+            13                2           0.3
invalid_coordinates            58               58           1.2


---
## Priority Scoring

SOP formula:
```
priority_score =
    category_importance
  + log(review_count)
  + rating_quality
  + distance_to_nearest_bike_station_bonus
  + geographic_coverage_bonus
  - duplicate_or_low_confidence_penalty
```

In [13]:
# Cell 7: Calculate priority scores for in-scope venues
in_scope = venues[~venues['venue_type'].isin(vs.OUT_OF_SCOPE_CATEGORIES)].copy()

# Build district label coverage dict (initially all 0 — will update after SerpApi)
district_label_coverage = {d: 0.0 for d in vs.DISTRICT_CENTERS.keys()}

# Merge citibike distance if available
if citibike_detail is not None and 'citibike_nearest_distance_m' in citibike_detail.columns:
    in_scope = in_scope.merge(
        citibike_detail[['venue_id', 'citibike_nearest_distance_m']],
        on='venue_id', how='left'
    )
else:
    in_scope['citibike_nearest_distance_m'] = None

# Calculate scores
scores = []
for _, row in in_scope.iterrows():
    score = vs.calculate_priority_score(
        venue_type=row['venue_type'],
        district=row['district'],
        review_count=vs.get_review_count(row['name']),
        rating=row.get('rating'),
        citibike_distance_m=row.get('citibike_nearest_distance_m'),
        district_label_coverage=district_label_coverage,
    )
    scores.append(score)

in_scope['priority_score'] = scores
in_scope = in_scope.sort_values('priority_score', ascending=False)

print(f'Priority scores calculated for {len(in_scope)} in-scope venues')
print(f'\nScore distribution:')
print(in_scope['priority_score'].describe().to_string())
print(f'\nTop 10 candidates:')
print(in_scope[['name', 'venue_type', 'district', 'priority_score']].head(10).to_string(index=False))

Priority scores calculated for 1559 in-scope venues

Score distribution:
count    1559.000000
mean        6.684983
std         0.807703
min         3.497895
25%         6.166506
50%         6.900608
75%         7.354113
max         7.705027

Top 10 candidates:
                                name venue_type     district  priority_score
          HSS Ambulatory Care Clinic healthcare midtown_east        7.705027
Inwood Hill Park (Payson Playground)   restroom       uptown        7.703020
    Weill Cornell Medical Associates healthcare midtown_east        7.702872
          Alfred E. Smith Playground   restroom     downtown        7.702075
                    Vein Care Center healthcare     downtown        7.691965
  Laurance S Rockefeller OP Pavilion healthcare midtown_east        7.688866
                  Allen Street Malls   restroom     downtown        7.676072
       Queensboro Oval Indoor Tennis   restroom midtown_east        7.672834
          Louis Brandeis High School healthcar

---
## SerpApi Batch Discovery

**Strategy (SOP)**:
- Use Search API with `category + district` queries to batch-discover venues with `popular_times`
- Each search returns up to 20 results → costs 1 API call
- Match results to local venues by proximity (haversine < 100m)
- Do NOT call Place API per venue here

**API Budget**: 250 calls/month total. This workflow uses ~16 Search + ~100 Place = ~116 calls.

In [14]:
# Cell 8: SerpApi batch discovery
search_results = []

if DRY_RUN:
    print('=== DRY-RUN MODE ===')
    print('No SERPAPI_API_KEY set. Generating synthetic results for pipeline testing.')
    print('To run live: export SERPAPI_API_KEY=your_key_here\n')
    
    # Generate synthetic search results for testing the pipeline
    # Simulate: each category×district search finds some venues with popular_times
    np.random.seed(42)
    for cat_query, google_type, clearpath_type in vs.SERPAPI_SEARCH_CATEGORIES:
        for district in vs.DISTRICT_CENTERS.keys():
            # Get venues of this type in this district
            mask = (in_scope['venue_type'] == clearpath_type) & (in_scope['district'] == district)
            district_venues = in_scope[mask]
            
            # Simulate: 20-40% of healthcare venues have popular_times
            sample_size = min(5, len(district_venues))
            if sample_size == 0:
                continue
            
            sampled = district_venues.sample(n=sample_size, random_state=42)
            
            for _, venue in sampled.iterrows():
                has_pt = np.random.random() < 0.30  # 30% have popular_times
                result = vs.SerpApiSearchResult(
                    place_id=f'ChIJ_{venue["venue_id"][:10]}',
                    data_id=f'0x{hash(venue["venue_id"]) % (10**16):016x}',
                    name=venue['name'],
                    address=venue.get('address', ''),
                    latitude=venue['latitude'],
                    longitude=venue['longitude'],
                    rating=venue.get('rating'),
                    reviews=vs.get_review_count(venue['name']),
                    type=google_type,
                    has_popular_times=has_pt,
                    popular_times_summary={'synthetic': True} if has_pt else None,
                    search_query=cat_query,
                    search_category=clearpath_type,
                    search_district=district,
                )
                search_results.append(result)
    
    print(f'Synthetic results generated: {len(search_results)}')
    print(f'With popular_times (simulated): {sum(1 for r in search_results if r.has_popular_times)}')

else:
    print('=== LIVE MODE ===')
    print(f'Using SerpApi key: ...{SERPAPI_KEY[-4:]}')
    search_results = vs.batch_search_discovery(
        venues=in_scope,
        api_key=SERPAPI_KEY,
        output_dir=OUTPUT_DIR,
        log_fn=print,
    )

=== LIVE MODE ===
Using SerpApi key: ...2bb8
  Search: 'hospital' in uptown...
    → 20 results returned
  Search: 'hospital' in midtown_east...
    → 20 results returned
  Search: 'hospital' in midtown_west...
    → 20 results returned
  Search: 'hospital' in downtown...
    → 14 results returned
  Search: 'medical clinic' in uptown...
    → 20 results returned
  Search: 'medical clinic' in midtown_east...
    → 20 results returned
  Search: 'medical clinic' in midtown_west...
    → 20 results returned
  Search: 'medical clinic' in downtown...
    → 20 results returned
  Search: 'pharmacy' in uptown...
    → 20 results returned
  Search: 'pharmacy' in midtown_east...
    → 20 results returned
  Search: 'pharmacy' in midtown_west...
    → 20 results returned
  Search: 'pharmacy' in downtown...
    → 20 results returned
  Search: 'dentist office' in uptown...
    → 20 results returned
  Search: 'dentist office' in midtown_east...
    → 20 results returned
  Search: 'dentist office' in m

In [ ]:
# Cell 8b: Cross-match SerpApi search results to local DB venues
matched_candidate_rows = []

for r in search_results:
    matched = vs._find_matching_venues(
        in_scope,
        r.latitude,
        r.longitude,
        max_distance_m=100,
    )

    for _, venue in matched.iterrows():
        if venue["venue_type"] != r.search_category:
            continue

        matched_candidate_rows.append({
            "venue_id": venue["venue_id"],
            "venue_name": venue["name"],
            "venue_type": venue["venue_type"],
            "district": venue["district"],
            "serpapi_place_id": r.place_id,
            "serpapi_name": r.name,
            "serpapi_type": r.type,
            "rating": r.rating,
            "reviews": r.reviews or 0,
            "distance_m": venue.get("_distance_m"),
            "result_obj": r,
        })

matched_candidates_df = pd.DataFrame(matched_candidate_rows)

if matched_candidates_df.empty:
    print("No SerpApi candidates matched local DB venues within 100m + same venue_type.")
else:
    matched_candidates_df = (
        matched_candidates_df
        .sort_values(["reviews", "rating"], ascending=[False, False])
        .drop_duplicates(subset=["venue_id"], keep="first")
        .reset_index(drop=True)
    )

    print(f"Matched DB venues: {len(matched_candidates_df)}")
    print(f"Unique venue_ids: {matched_candidates_df['venue_id'].nunique()}")
    print(f"Unique place_ids: {matched_candidates_df['serpapi_place_id'].nunique()}")
    print(matched_candidates_df[[
        "venue_id", "venue_name", "district", "serpapi_name",
        "serpapi_type", "reviews", "rating", "distance_m"
    ]].head(20).to_string(index=False))


In [ ]:
# Cell 9: Search results analysis
if search_results:
    results_df = pd.DataFrame([vars(r) for r in search_results])
    print(f'\nSearch Results Summary:')
    print(f'  Total discovered: {len(results_df)}')
    print(f'  With popular_times: {results_df["has_popular_times"].sum()}')
    print(f'  Without popular_times: {(~results_df["has_popular_times"]).sum()}')
    print(f'\nBy category:')
    print(results_df.groupby('search_category')['has_popular_times'].agg(['count', 'sum']).to_string())
    print(f'\nBy district:')
    print(results_df.groupby('search_district')['has_popular_times'].agg(['count', 'sum']).to_string())
else:
    print('No search results (check API key or try dry-run mode)')

---
## Place API Validation (Limited)

Only for the top candidates. In dry-run mode, this is simulated.

In [ ]:
# Cell 10: Place API validation (top candidates only)
if DRY_RUN:
    print('=== DRY-RUN: Skipping Place API validation ===')
    validated_results = search_results  # use search results as-is
else:
    # Only validate the most promising candidates (those with popular_times from search)
    candidates_to_validate = [r for r in search_results if r.has_popular_times]
    print(f'Validating {len(candidates_to_validate)} candidates via Place API...')
    validated_results = vs.validate_candidates_with_place_api(
        candidates=candidates_to_validate,
        api_key=SERPAPI_KEY,
        output_dir=OUTPUT_DIR,
        max_calls=100,
        log_fn=print,
    )

---
## Label Status & ML Candidate List

In [ ]:
# Cell 11: Generate label status for all venues
label_status_df = vs.generate_label_status(
    venues=venues,
    search_results=validated_results,
    citibike_detail=citibike_detail,
)

print(f'Label Status Summary:')
print(f'  Total venues: {len(label_status_df)}')
print(f'\n  By label_status:')
print(label_status_df['label_status'].value_counts().to_string())
print(f'\n  ML eligible: {label_status_df["ml_eligible"].sum()}')
print(f'  Not eligible: {(~label_status_df["ml_eligible"]).sum()}')
print(f'\n  By prediction_source:')
print(label_status_df['prediction_source'].value_counts().to_string())

In [ ]:
# Cell 12: Update category and district audits with label status
cat_audit_final = vs.audit_coverage_by_category(venues, label_status_df)
dist_audit_final = vs.audit_coverage_by_district(venues, label_status_df)

print('Category Coverage (with label status):')
print(cat_audit_final.to_string(index=False))
print(f'\nDistrict Coverage (with label status):')
print(dist_audit_final.to_string(index=False))

In [ ]:
# Cell 13: Generate ML candidate list
candidates_path = OUTPUT_DIR / 'venue_ml_candidates.csv'
candidates = vs.generate_candidate_list(label_status_df, candidates_path)
print(f'ML Candidates: {len(candidates)} venues')
print(f'Saved to: {candidates_path}')
if len(candidates) > 0:
    print(f'\nTop 10 ML candidates:')
    print(candidates[['venue_id', 'name', 'venue_type', 'district', 'serpapi_place_id']].head(10).to_string(index=False))

In [ ]:
# Cell 14: Save label status CSV
label_status_path = OUTPUT_DIR / 'venue_label_status.csv'
label_status_df.to_csv(label_status_path, index=False)
print(f'Label status saved to: {label_status_path}')
print(f'  Rows: {len(label_status_df)}')
print(f'  Columns: {list(label_status_df.columns)}')

In [ ]:
# Cell 15: Generate coverage audit report
report_path = OUTPUT_DIR / 'coverage_audit_report.md'
report = vs.generate_coverage_report(
    category_audit=cat_audit_final,
    district_audit=dist_audit_final,
    citibike_audit=cb_audit,
    label_status_df=label_status_df,
    search_results=search_results,
    output_path=report_path,
)
print(f'Report saved to: {report_path}')
print(f'\n--- Report Preview (first 50 lines) ---')
for line in report.split('\n')[:50]:
    print(line)

In [ ]:
# Cell 16: Save run metadata
meta = vs.save_run_metadata(
    venues=venues,
    search_results=search_results,
    label_status_df=label_status_df,
    output_dir=OUTPUT_DIR,
    api_calls_search=0 if DRY_RUN else len(set(r.search_query + r.search_district for r in search_results)),
    api_calls_place=0 if DRY_RUN else len([r for r in search_results if r.has_popular_times]),
)
print(f'Run metadata saved to: {OUTPUT_DIR / "run_metadata.json"}')
print(json.dumps(meta, indent=2))

In [ ]:
# Cell 17: Final summary — all output artifacts
print('=' * 60)
print('VENUE ML COVERAGE WALKTHROUGH — COMPLETE')
print('=' * 60)
print(f'\nOutput directory: {OUTPUT_DIR}')
print(f'\nArtifacts generated:')
for f in sorted(OUTPUT_DIR.rglob('*')):
    if f.is_file():
        size = f.stat().st_size
        print(f'  {f.relative_to(OUTPUT_DIR):<45} {size:>8,} bytes')

print(f'\nKey metrics:')
print(f'  Total venues:          {len(venues):,}')
print(f'  In-scope (SerpApi screening): {len(venues[~venues["venue_type"].isin(vs.OUT_OF_SCOPE_CATEGORIES)]):,}')
print(f'  Out-of-scope (AED):    {len(venues[venues["venue_type"].isin(vs.OUT_OF_SCOPE_CATEGORIES)]):,}')
print(f'  ML eligible:           {label_status_df["ml_eligible"].sum():,}')
print(f'  prediction_source=ml_model:  {(label_status_df["prediction_source"] == "ml_model").sum():,}')
print(f'  prediction_source=rule_fallback: {(label_status_df["prediction_source"] == "rule_fallback").sum():,}')
print(f'  prediction_source=none:      {(label_status_df["prediction_source"] == "none").sum():,}')
print(f'\nMode: {"DRY-RUN (synthetic data)" if DRY_RUN else "LIVE (real SerpApi data)"}')
print(f'Completed: {datetime.now():%Y-%m-%d %H:%M}')